# 01a · Build the Pleroma dataset

Reads the crawled Pleroma conversations (`pleroma_conversations.zip`, one pickled
conversation per file, grouped by the instance it was crawled from) and writes:

| File | Content |
|---|---|
| `instances.csv` | `instance_id ↔ instance_uri` |
| `authors.csv` | `author_id ↔ author_username` (`user@domain`) |
| `toots.csv` | `toot_id ↔ toot_uri` |
| `conversations.csv` | `conversation_id ↔ conversation_pkl_filename, conversation_uri` |
| `pleroma_toots_object.csv` | one row per toot observation |
| `replies.csv` | one row per reply edge (`toot_id_child → toot_id_parent`) |
| `failed_files.csv` | conversation files that could not be processed |

IDs are 1-based and assigned by lexicographic order of the original values, so the
same input always produces the same IDs. The mapping files are the single source of
truth for IDs: every later step reloads them instead of regenerating them.

In [ ]:
import csv
import gc
import json
import pickle
import zipfile
from datetime import datetime
from pathlib import Path, PurePosixPath

import pandas as pd
from tqdm.auto import tqdm

## Configuration

In [ ]:
DATA_DIR = Path("../../data/pleroma")
ZIP_PATH = DATA_DIR / "raw" / "pleroma_conversations.zip"

INSTANCES_CSV = DATA_DIR / "instances.csv"
AUTHORS_CSV = DATA_DIR / "authors.csv"
TOOTS_CSV = DATA_DIR / "toots.csv"
CONVERSATIONS_CSV = DATA_DIR / "conversations.csv"
TOOTS_OBJECT_CSV = DATA_DIR / "pleroma_toots_object.csv"
REPLIES_CSV = DATA_DIR / "replies.csv"
FAILED_FILES_CSV = DATA_DIR / "failed_files.csv"

# Folder inside the archive that contains one sub-folder per crawled instance.
ARCHIVE_ROOT = "instances_conv_graphs/"

DATA_DIR.mkdir(parents=True, exist_ok=True)
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Archive not found: {ZIP_PATH}")

## Helper functions

In [ ]:
def nonempty_string(value):
    """Stripped string, or None for a missing/empty value."""
    if value is None:
        return None
    value = str(value).strip()
    return value or None


def instance_found_from_member(member_name):
    """Instance a conversation was crawled from, taken from its path in the archive."""
    parent = PurePosixPath(member_name).parent.as_posix()
    return parent[len(ARCHIVE_ROOT):] if parent.startswith(ARCHIVE_ROOT) else parent


def conversation_filename(member_name):
    """Pickle filename, used as the conversation mapping key."""
    return PurePosixPath(member_name).name


def conversation_uri_from_filename(filename):
    """Root-toot URI encoded in the pickle filename."""
    return filename.replace("\\", "/").removesuffix(".pkl")


def normalize_uri_for_root_comparison(uri):
    if uri is None:
        return ""
    return str(uri).replace("\\", "/").removeprefix("https://").rstrip("/")


def normalize_author(author_value, instance_found):
    """
    Canonical ``username@domain`` author name.

    Remote authors already carry their domain (``alice@example.social``). Local
    authors (``alice``) belong to the instance the conversation was crawled from.
    """
    author = nonempty_string(author_value)
    instance_found = nonempty_string(instance_found)
    if author is None:
        return None
    if "@" in author:
        return author
    if instance_found is not None:
        return f"{author}@{instance_found}"
    return author


def parse_created_at(value):
    """ISO timestamp; the original value is kept if it cannot be parsed."""
    value = nonempty_string(value)
    if value is None:
        return ""
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00")).isoformat()
    except (TypeError, ValueError):
        return value


def json_or_empty(value):
    if value is None or value == "":
        return ""
    return json.dumps(value, ensure_ascii=False)


def add_unique_value(values, value):
    value = nonempty_string(value)
    if value is not None:
        values.add(value)
    return value


def build_sorted_mapping(values):
    """Stable 1-based IDs in lexicographic order."""
    return {value: i for i, value in enumerate(sorted(values), start=1)}


def write_mapping(path, rows, header):
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle, quoting=csv.QUOTE_MINIMAL)
        writer.writerow(header)
        writer.writerows(rows)

## List the conversation files

Member names are sorted so that every pass traverses the conversations in the same order.

In [ ]:
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    pkl_files = sorted(
        info.filename
        for info in archive.infolist()
        if not info.is_dir() and info.filename.lower().endswith(".pkl")
    )

if not pkl_files:
    raise RuntimeError("No .pkl conversation files found in the archive.")

print(f"Conversation files: {len(pkl_files):,}")

## Pass 1: collect unique values and build the ID mappings

In [ ]:
instance_values, author_values, toot_values, conversation_values = set(), set(), set(), set()
failures = []

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for member_name in tqdm(pkl_files, desc="Pass 1", unit="file"):
        found_instance = instance_found_from_member(member_name)
        add_unique_value(instance_values, found_instance)
        add_unique_value(conversation_values, conversation_filename(member_name))

        try:
            with archive.open(member_name, "r") as handle:
                conversation = pickle.load(handle)

            for toot_uri, node_data in (conversation.get("nodes", {}) or {}).items():
                add_unique_value(toot_values, toot_uri)
                add_unique_value(author_values, normalize_author(node_data.get("author"), found_instance))
                add_unique_value(instance_values, node_data.get("instance"))
        except Exception as exc:
            failures.append((member_name, "pass_1", repr(exc)))

instance_map = build_sorted_mapping(instance_values)
author_map = build_sorted_mapping(author_values)
toot_map = build_sorted_mapping(toot_values)
conversation_map = build_sorted_mapping(conversation_values)

print(f"Instances:     {len(instance_map):,}")
print(f"Authors:       {len(author_map):,}")
print(f"Toots:         {len(toot_map):,}")
print(f"Conversations: {len(conversation_map):,}")
print(f"Failures:      {len(failures):,}")

In [ ]:
write_mapping(INSTANCES_CSV, ((i, v) for v, i in instance_map.items()), ["instance_id", "instance_name"])
write_mapping(AUTHORS_CSV, ((i, v) for v, i in author_map.items()), ["author_id", "author_username"])
write_mapping(TOOTS_CSV, ((i, v) for v, i in toot_map.items()), ["toot_id", "toot_uri"])
write_mapping(
    CONVERSATIONS_CSV,
    ((i, f, conversation_uri_from_filename(f)) for f, i in conversation_map.items()),
    ["conversation_id", "conversation_pkl_filename", "conversation_uri"],
)

## Pass 2: toot observations

Each toot in each crawled conversation is one row, so the same toot can appear more
than once when it was observed through several instances or conversations.

In [ ]:
TOOT_COLUMNS = [
    "created_at",
    "text",
    "author_id",
    "instance_toot_belong_id",
    "conversation_id",
    "toot_id",
    "instance_toot_found_id",
    "reblogged",
    "reblogs_count",
    "old_conv_ids",
    "root_toot",
]

rows_written = 0
rows_missing_mapping = 0

with (
    zipfile.ZipFile(ZIP_PATH, "r") as archive,
    TOOTS_OBJECT_CSV.open("w", newline="", encoding="utf-8") as output_handle,
):
    writer = csv.DictWriter(output_handle, fieldnames=TOOT_COLUMNS, quoting=csv.QUOTE_ALL)
    writer.writeheader()

    for member_name in tqdm(pkl_files, desc="Pass 2", unit="file"):
        try:
            with archive.open(member_name, "r") as handle:
                conversation = pickle.load(handle)

            nodes = conversation.get("nodes", {}) or {}
            found_instance = nonempty_string(instance_found_from_member(member_name))
            found_instance_id = instance_map.get(found_instance)

            conv_filename = conversation_filename(member_name)
            conversation_id = conversation_map.get(conv_filename)
            root_uri = normalize_uri_for_root_comparison(conversation_uri_from_filename(conv_filename))

            for toot_uri in sorted(nodes):
                node_data = nodes[toot_uri]
                toot_uri_clean = nonempty_string(toot_uri)

                author_id = author_map.get(normalize_author(node_data.get("author"), found_instance))
                belonging_instance_id = instance_map.get(nonempty_string(node_data.get("instance")))
                toot_id = toot_map.get(toot_uri_clean)

                ids = (author_id, belonging_instance_id, conversation_id, toot_id, found_instance_id)
                if any(v is None for v in ids):
                    rows_missing_mapping += 1
                    continue

                writer.writerow({
                    "created_at": parse_created_at(node_data.get("created_at")),
                    "text": node_data.get("text", "") or "",
                    "author_id": author_id,
                    "instance_toot_belong_id": belonging_instance_id,
                    "conversation_id": conversation_id,
                    "toot_id": toot_id,
                    "instance_toot_found_id": found_instance_id,
                    "reblogged": bool(node_data.get("reblogged", False)),
                    "reblogs_count": node_data.get("reblogs_count", 0) or 0,
                    "old_conv_ids": json_or_empty(node_data.get("old_conv_ids")),
                    "root_toot": normalize_uri_for_root_comparison(toot_uri_clean) == root_uri,
                })
                rows_written += 1

        except Exception as exc:
            failures.append((member_name, "pass_2", repr(exc)))

        if rows_written and rows_written % 100_000 == 0:
            output_handle.flush()
            gc.collect()

print(f"Rows written:          {rows_written:,}")
print(f"Rows missing a mapping: {rows_missing_mapping:,}")
print(f"Failures:              {len(failures):,}")

In [ ]:
with FAILED_FILES_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["file_name", "processing_stage", "error"])
    writer.writerows(failures)

pd.read_csv(TOOTS_OBJECT_CSV, nrows=5)

## Reply edges

Each conversation pickle stores `edges` as `{child_uri: parent_uri}`. Every edge is
written as one row of `replies.csv`. A child without a parent (the root) is written
as a self-edge.

The next cell reloads the saved mappings, so this section can also be run on its own
in a fresh kernel (after the configuration and helper cells).

In [ ]:
def load_mapping(csv_path, id_col, value_col):
    d = pd.read_csv(csv_path, dtype=str)
    return dict(zip(d[value_col], d[id_col].astype(int)))


instance_map = load_mapping(INSTANCES_CSV, "instance_id", "instance_name")
toot_map = load_mapping(TOOTS_CSV, "toot_id", "toot_uri")
conversation_map = load_mapping(CONVERSATIONS_CSV, "conversation_id", "conversation_pkl_filename")

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    pkl_files = sorted(
        info.filename
        for info in archive.infolist()
        if not info.is_dir() and info.filename.lower().endswith(".pkl")
    )

In [ ]:
reply_id = 1
counts = dict.fromkeys(
    ["files_processed", "files_failed", "empty_edges", "missing_instance",
     "missing_conversation", "missing_child", "missing_parent"],
    0,
)
failed_examples = []

with (
    zipfile.ZipFile(ZIP_PATH, "r") as archive,
    REPLIES_CSV.open("w", newline="", encoding="utf-8") as output_file,
):
    writer = csv.writer(output_file)
    writer.writerow(["instance_toot_found_id", "conversation_id", "reply_id", "toot_id_child", "toot_id_parent"])

    for member_name in tqdm(pkl_files, desc="Reply edges", unit="file"):
        try:
            instance_id = instance_map.get(instance_found_from_member(member_name))
            if instance_id is None:
                counts["missing_instance"] += 1
                continue

            conversation_id = conversation_map.get(conversation_filename(member_name))
            if conversation_id is None:
                counts["missing_conversation"] += 1
                continue

            with archive.open(member_name, "r") as pkl_file:
                conversation = pickle.load(pkl_file)
            if not isinstance(conversation, dict):
                raise TypeError(f"Expected dict, got {type(conversation).__name__}")

            edges = conversation.get("edges", {}) or {}
            if not isinstance(edges, dict):
                raise TypeError(f"Expected edges to be dict, got {type(edges).__name__}")

            counts["files_processed"] += 1
            if not edges:
                counts["empty_edges"] += 1
                continue

            for child_uri in sorted(edges):
                child_id = toot_map.get(nonempty_string(child_uri))
                if child_id is None:
                    counts["missing_child"] += 1
                    continue

                parent_uri = nonempty_string(edges[child_uri])
                if parent_uri is None:
                    parent_id = child_id
                else:
                    parent_id = toot_map.get(parent_uri)
                    if parent_id is None:
                        counts["missing_parent"] += 1
                        continue

                writer.writerow([instance_id, conversation_id, reply_id, child_id, parent_id])
                reply_id += 1

        except Exception as error:
            counts["files_failed"] += 1
            if len(failed_examples) < 20:
                failed_examples.append({"file": member_name, "error": repr(error)})

print(f"Reply edges written: {reply_id - 1:,}")
for key, value in counts.items():
    print(f"{key:22s} {value:,}")
for item in failed_examples:
    print(item)